# Download data and extract Perch embeddings

Run once before training notebooks.


In [ ]:
from google.colab import drive
import os
import sys
from pathlib import Path

drive.mount("/content/drive")

REPRO_ROOT = Path("/content/drive/MyDrive/BirdCLEF_Project/repro")
sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)

!pip install -q kaggle onnxruntime-gpu soundfile tqdm

from birdclef.colab import mount_and_configure, set_kaggle_token
from birdclef.paths import DRIVE_EMBEDDINGS_DIR, DRIVE_EMBEDDINGS_TTA_DIR, DRIVE_ROOT, PERCH_ONNX

mount_and_configure()
set_kaggle_token()
DRIVE_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_EMBEDDINGS_TTA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess
from pathlib import Path

RAW = Path("/content/bird_data")
RAW.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["kaggle", "competitions", "download", "-c", "birdclef-2026", "-p", str(RAW)],
    check=True,
)
zip_path = RAW / "birdclef-2026.zip"
if zip_path.exists():
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(RAW)], check=True)

import shutil
for name in ("train.csv", "sample_submission.csv"):
    src = RAW / name
    if src.exists():
        shutil.copy(src, Path("/content") / name)
        shutil.copy(src, repro_root / "data" / "metadata" / name)
        print(f"Staged {name}")


## Perch ONNX model


In [ ]:
from birdclef.paths import PERCH_ONNX

if not PERCH_ONNX.exists():
    raise FileNotFoundError(
        f"Place perch_v2_no_dft.onnx at {PERCH_ONNX}."
    )
print(f"Found ONNX model at {PERCH_ONNX}")


## Baseline embeddings (5 s windows)


In [ ]:
import pandas as pd
from birdclef.extract import extract_baseline_embeddings
from birdclef.paths import DRIVE_EMBEDDINGS_DIR, PERCH_ONNX, TRAIN_AUDIO_DIR, TRAIN_CSV

save_dir = str(DRIVE_EMBEDDINGS_DIR)

df = pd.read_csv(TRAIN_CSV)
extract_baseline_embeddings(df, str(PERCH_ONNX), str(TRAIN_AUDIO_DIR), save_dir)
print(f"Baseline embeddings saved to {save_dir}")



## TTA embeddings (2.5 s stride)


In [ ]:
from birdclef.extract import extract_tta_embeddings
from birdclef.paths import DRIVE_EMBEDDINGS_TTA_DIR, PERCH_ONNX, TRAIN_AUDIO_DIR

save_dir = str(DRIVE_EMBEDDINGS_TTA_DIR)

extract_tta_embeddings(df, str(PERCH_ONNX), str(TRAIN_AUDIO_DIR), save_dir)
print(f"TTA embeddings saved to {save_dir}")



## Optional: archive embeddings to Drive

Skip if archives already exist. Re-run after a fresh extraction to rebuild zips.


In [ ]:
import shutil
from pathlib import Path

for folder, archive_name in (
    (DRIVE_EMBEDDINGS_DIR, "embeddings_v2_archive.zip"),
    (DRIVE_EMBEDDINGS_TTA_DIR, "embeddings_v2_TTA_archive.zip"),
):
    local_zip = Path("/content") / archive_name.replace(".zip", "")
    shutil.make_archive(str(local_zip), "zip", str(folder))
    shutil.move(str(local_zip) + ".zip", str(DRIVE_ROOT / archive_name))
    print(f"Archived {folder} to {DRIVE_ROOT / archive_name}")
